In [79]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.utils import bootstrap_project_paths

project_root = bootstrap_project_paths()

from src.data import load_sales
from src.pipelines import train_validate_models
from src.features import one_hot_encode, add_lag_features, add_rolling_features
from src.models import *
import pandas as pd
import numpy as np

DATA_ROOT= project_root / "data/datathon-2026-round-1"

Project root: D:\MyML\datathon2026
Source path added: D:\MyML\datathon2026\src


In [121]:
TRAIN_RANGE = ("2013-01-01","2021-12-31")  # Date range for training data
VALIDATION_RANGE = ("2022-01-01","2022-12-31")  # Date range for validation data

df = load_sales(data_root=DATA_ROOT)
# Basic time features
df["day"] = df["date"].dt.day
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year

# day_of_month, isweekend không không thay dổi performance 
# nhưng không thể thay thế 2 feature trên
df["day_of_week"] = df["date"].dt.dayofweek
df["week_of_year"] = df["date"].dt.isocalendar().week


# Cái này quan trọng
df["month_sin"] = np.sin(2 * np.pi * df["month"]/12)
df["month_cos"] = np.cos(2 * np.pi * df["month"]/12)

# Cái feature này thần kỳ vcl
df["is_month8_odd"] = ((df["month"] == 8) & (df["year"] % 2 == 1))

# Cần giải thích tại sao lag 30 cho revenue lại tệ
# 11 mạnh vượt trội, 12, 13, 14 cải thiện nhưng không lấy 
# 21, 28, 30, 10 làm tệ đi
df = add_lag_features(df, lags=[1,7,11], target_col="Revenue", date_col="date")

# Cần giải thích tại sao rolling 7, 30 cho revenue lại tệ
# df = add_rolling_features(df, windows=[7, 30], target_col="Revenue", date_col="date")

model_config = SklearnRegressorConfig(
    model_type="lightgbm"
)
results = train_validate_models(
    df,
    train_range=TRAIN_RANGE,
    predict_range=VALIDATION_RANGE,
    model_config=model_config
)
results["metrics"]

{'Revenue': {'mae': 520952.65419360483,
  'rmse': 732237.6182616778,
  'mape': 18.381617528713118,
  'smape': 17.78747696592552,
  'r2': 0.8086220917214222},
 'COGS': {'mae': 489856.51761645946,
  'rmse': 665166.0713916149,
  'mape': 19.742494225093584,
  'smape': 19.178623771145215,
  'r2': 0.7920268919574528}}

In [ ]:
# 3 biến tạm này dở ẹt
day_of_month = df["date"].dt.day
day_to_month_end = df["date"].dt.days_in_month - day_of_month
dist_to_payday = np.minimum(day_of_month, day_to_month_end)  

# mae giảm 0.2%, tăng các cái còn lại, cần xem lại
# salary_heat = np.where(dist_to_payday <= 3, 1 / (dist_to_payday + 1), 0.0)
# is_salary_period = (dist_to_payday <= 3).astype(int)
# df["payday_weekend"] = (is_salary_period & (df["day_of_week"] >= 4)).astype(int)

model_config = SklearnRegressorConfig(
    model_type="lightgbm"
)
results = train_validate_models(
    df,
    train_range=TRAIN_RANGE,
    predict_range=VALIDATION_RANGE,
    model_config=model_config
)
results["metrics"]

In [ ]:

model_config = SklearnRegressorConfig(
    model_type="lightgbm"
)
results = train_validate_models(
    df,
    train_range=TRAIN_RANGE,
    predict_range=VALIDATION_RANGE,
    model_config=model_config
)
results["metrics"]

{'Revenue': {'mae': 580362.5654506256,
  'rmse': 814001.8648100265,
  'mape': 20.626541592136196,
  'smape': 19.183407582445376,
  'r2': 0.7634959785831261},
 'COGS': {'mae': 497960.5153749969,
  'rmse': 690856.2131899564,
  'mape': 20.417977899768605,
  'smape': 19.083292447805764,
  'r2': 0.7756519287537018}}